# NLP 2026
# Lab 3: Classification and BERT

Have you ever read a movie review and wondered:

```“Is this review actually positive or negative?” 🤔```

In this lab, you will build your own sentiment analysis tool using Natural Language Processing (NLP)! Your goal is to automatically classify movie reviews into one of two categories:

✅ Positive

❌ Negative

We will approach this as a binary classification task and you will experiment with increasingly powerful methods — from classic machine learning to modern neural networks based on transformers 🚀

### 🎯 Learning Goals ###

By completing this lab, you should be able to:

- Formulate sentiment analysis as a binary classification problem

- Design and evaluate hand-crafted text features

- Implement a Bag-of-Words representation

- Apply and evaluate Logistic Regression and alternative classifiers

- Understand how BERT tokenization and embeddings work

- Extract sentence representations using: ```CLS``` token, mean token pooling

- Compare classical ML and transformer-based methods

- Critically analyze evaluation metrics beyond accuracy 📊



### Score breakdown

| Exercise            | Points |
|---------------------|--------|
| [Exercise 1](#e1)   | 1      |
| [Exercise 2](#e2)   | 5      |
| [Exercise 3](#e3)   | 5      |
| [Exercise 4](#e4)   | 5      |
| [Exercise 5](#e5)   | 5      |
| [Exercise 6](#e6)   | 5      |
| [Exercise 7](#e7)   | 3      |
| [Exercise 8](#e8)   | 6      |
| [Exercise 9](#e9)   | 5      |
| [Exercise 10](#e10) | 5      |
| [Exercise 12](#e12) | 5      |
| [Exercise 13](#e13) | 10     |
| Total               | 60     |

This score will be scaled down to 0.6 and that will be your final lab score.

### 📌 **Instructions for Delivery** (📅 **Deadline: 23/Feb 18:00**, 🎭 *wildcards possible*)

✅ **Submission Requirements**
+ 📄 You need to submit a **notebook** 📓 with the code, appropriate comments and figures in all questions. Make sure to have a mix of code (some explanations needed if not clear what you implement), figures to support the answers or your claims and proper amount of text to explain your reasoning, answer etc.
+ ⚡ Make sure that **all cells are executed properly** ⚙️ and that **all figures/results/plots** 📊 you include in the report are also visible in your **executed notebook**.
+ You can work on Google Collab (or other environments), but you need to make sure that your delivered notebook is executed properly.

✅ **Collaboration & Integrity**
+ 🗣️ While you may **discuss** the lab with others, you must **write your solutions with your group only**. If you **discuss specific tasks** with others, please **include their names** below.
+ 📜 **Honor Code applies** to this lab. For more details, check **Syllabus §7.2** ⚖️.
+ 📢 **Mandatory Disclosure**:
   - Any **websites** 🌐 (e.g., **Stack Overflow** 💡) or **other resources** used must be **listed and disclosed**.
   - Any **GenAI tools** 🤖 (e.g., **ChatGPT**) used must be **explicitly mentioned**.
   - 🚨 **Failure to disclose these resources is a violation of academic integrity**. See **Syllabus §7.3** for details.

## 0. Setup

We first install the scikit-learn library [Scikit-learn](https://scikit-learn.org/stable/). We will use its classification models.

In [1]:
!pip install -U scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 50.2 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


We will need [PyTorch](https://pytorch.org/) installed. It is a very popular deep learning library that offers modularized versions of many of the sequence models we discussed in class. It's an important tool that you may want to practice further if you want to dive deeper into NLP, since much of the current academic and industrial research uses it.

Some resources to look further are given below.

* [Documentation](https://pytorch.org/docs/stable/index.html) (We will need this soon)

* [Installation Instructions](https://pytorch.org/get-started/locally/)

* [Quickstart Tutorial](https://pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html)

The cell below should install the library:

In [2]:
!pip install torch torchvision

The last bit we need is the huggingface transformers library (here is the documentation [https://huggingface.co/docs/transformers/en/index](https://huggingface.co/docs/transformers/en/index)). Transformers are one of the most influential architectures in handling sequences (not only in language). As we discussed in lectures, they excel at taking into account context (which is the salt-and-pepper of NLP) with mechansisms such as self-attetion, which allows them to weigh the importance of different words in a sentence. If you want to know more, revisit the course material (slides and textbook).

We already used huggingface datasets in previous labs and huggingface transformers integrates nicely with that. Apart from the ease of use, huggingface is also providing pre-trained models of different kinds. The list can be found [here](https://huggingface.co/models) ([https://huggingface.co/models](https://huggingface.co/models)). The following line should be enough to install huggingface transformers library:

In [3]:
!pip install transformers

Here, we import the libraries:

In [4]:
import re
from collections import Counter

import datasets
import numpy as np
import torch
import tqdm
import transformers
import html
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    accuracy_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import ParameterGrid
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from torch.utils.data.dataloader import DataLoader

## 1. Loading the Dataset

We will work with the IMDB dataset [https://huggingface.co/datasets/stanfordnlp/imdb](https://huggingface.co/datasets/stanfordnlp/imdb). It contains the reviews and a label that indicates whether the review is positive or not (the neutral reviews have been filtered out). You can read the paper [here](https://aclanthology.org/P11-1015/).

In [5]:
dataset = datasets.load_dataset('stanfordnlp/imdb', split=['train', 'test'])
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

[Dataset({
    features: ['text', 'label'],
    num_rows: 25000
}), Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})]


Notice that the dataset has been loaded as a list of two datasets. They are the `train` and `test` splits that we asked for.
We will use the validation subset to tune the parameters. So, let's split the `train` subset and create a `DatasetDict` object:

In [6]:
train_valid_split = dataset[0].train_test_split(5000)
dataset = datasets.DatasetDict({
    'train': train_valid_split['train'].shuffle(),
    'validation': train_valid_split['test'].shuffle(),
    'test': dataset[1].shuffle(),
})
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})


We can print several examples from the `train` dataset:

In [7]:
for i in range(5):
    print('i', i)
    print(dataset['train'][i]['text'])
    print(dataset['train'][i]['label'])
    print()

i 0
I have never seen one of these SciFi originals before, this was the first. I think it only fair to judge the acting, direction/production, set design and even the CGI effects on the other SciFi movies. To compare it to your typical Hollywood production is unfair. I will say, however, that overall Aztec Rex was not exactly reminiscent of Werner Herzog's masterpiece Aguirre, Wrath of God.<br /><br />I will begin by noting that, yes, I do recognize the fact that this movie has more to do with culture-clash than it does with dinosaurs. Despite this being a made-for-TV sci-fi movie, there is some underlying context to the story which I shall examine. The symbolic elements included are evident enough.<br /><br />Consequently, as a student of history, theology, mythology and film: I found the dialogue outrageous and the plot themes to be somewhat insulting. I am not asking for any mea culpas on behalf of the producers - as I said before the movie is what it is. But what concerns me is tha

Let's extract the labels from the dataset. We will use them to train and evaluate our classifiers.

In [8]:
y_train = dataset['train']['label']
print(y_train)
y_valid = dataset['validation']['label']
print(y_valid)

Column([0, 0, 0, 1, 1])
Column([0, 0, 0, 1, 0])


<a name='e1'></a>
#### Exercise 1: Cleaning the text

(1p) In this exercise you should clean the text in the dataset. This is the same step we saw in the previous labs.

If you think this step is not necessary in this use case, you can skip this step, but make sure to justify your decision.

In [9]:
def clean(text):
    """
    Cleans the text
    Args:
        text: a string that will be cleaned

    Returns: the cleaned text

    """

    # Empty text
    if text == '':
        return text

    ### YOUR CODE HERE
    # Replace common IMDB HTML line breaks
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE)

    # Remove any remaining HTML tags
    text = re.sub(r"</?[^>]+>", " ", text)

    # Unescape entities like &quot; &amp;
    text = html.unescape(text)

    # Remove control characters
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    ### YOUR CODE ENDS HERE

    return text


def clean_example(example):
    """
    Applies the clean() function to the example from the Dataset
    Args:
        example: an example from the Dataset

    Returns: update example with cleaned 'text' column

    """
    example['text'] = clean(example['text'])
    return example


dataset = dataset.map(clean_example, desc="Cleaning")
print(dataset)

Cleaning:   0%|          | 0/20000 [00:00<?, ? examples/s]

Cleaning:   0%|          | 0/5000 [00:00<?, ? examples/s]

Cleaning:   0%|          | 0/25000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})


## 2. Hand-crafted Features

<a name='e2'></a>
#### Exercise 2: Hand-crafted features

(5p) Write your own hand-crafted feature extraction function. Include at least these types of features:
- length of the text,
- number of different punctuation characters,
- number of positive and negative words.

In [10]:
### YOUR CODE HERE
# you can define the positive and negative words here
POSITIVE_WORDS = {
    "good","great","excellent","amazing","awesome","fantastic","wonderful","best","brilliant",
    "love","loved","lovely","enjoy","enjoyed","enjoyable","perfect","favorite","favourite",
    "beautiful","fun","funny","hilarious","superb","outstanding","impressive","masterpiece",
    "charming","delightful","smart","strong","engaging","touching","heartwarming","cool",
    "recommend","recommended","recommendation","satisfying"
}

NEGATIVE_WORDS = {
    "bad","terrible","awful","horrible","worst","boring","dull","poor","waste","stupid",
    "hate","hated","annoying","disappointing","disappointed","disaster","pathetic","weak",
    "mediocre","ridiculous","lame","mess","unwatchable","predictable","nonsense","crap",
    "garbage","painful","problem","fails","failed","failure","slow","confusing"
}

_NEGATORS = {"not", "no", "never", "none", "n't", "hardly", "rarely", "seldom"}
_word_re = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+")
_punct_re = re.compile(r"[^\w\s]")


def calculate_features(text):
    features = []
    ### YOUR CODE HERE
    text = text or ""
    tokens = _word_re.findall(text.lower())

    # 1) length of the text (use log to reduce scale dominance)
    num_chars = len(text)
    num_tokens = len(tokens)
    features.append(np.log1p(num_chars))
    features.append(np.log1p(num_tokens))

    # 2) number of different punctuation characters
    puncts = _punct_re.findall(text)
    features.append(float(len(set(puncts))))

    # 3) number of positive and negative words
    pos = sum(1 for t in tokens if t in POSITIVE_WORDS)
    neg = sum(1 for t in tokens if t in NEGATIVE_WORDS)
    features.append(float(pos))
    features.append(float(neg))

    # A couple of cheap-but-useful extras (still “hand-crafted”)
    features.append(float(pos - neg))  # polarity
    features.append(float(text.count("!")))
    features.append(float(text.count("?")))
    features.append(float(sum(1 for t in tokens if t in _NEGATORS)))  # negation presence/count

    # ratio-ish features
    denom = (num_tokens + 1.0)
    features.append(float(pos) / denom)
    features.append(float(neg) / denom)
    ### YOUR CODE ENDS HERE
    return np.array(features, dtype=float)


### YOUR CODE ENDS HERE

The function below will apply your feature extraction implementation to a specified dataset.

In [11]:
def calculate_features_dataset(dataset, features_fn):
    all_features = []
    for e in tqdm.tqdm(dataset, desc='Extracting features'):
        text = e['text']
        features = features_fn(text)
        all_features.append(features)
    all_features = np.array(all_features, dtype=float)
    return all_features

And we can obtain the features for the `train` and `validation` splits. Later you will need to do the same for the `test` subset.

In [12]:
X_train = calculate_features_dataset(dataset['train'], calculate_features)
X_valid = calculate_features_dataset(dataset['validation'], calculate_features)

Extracting features: 100%|██████████| 5000/5000 [00:01<00:00, 4934.68it/s]


### 2.1 Classification

In this section, we will create and train a logistic regression classifier. We will train it on the `train` subset and evaluate on the `validation` split. Later, you will do a final comparison between methods on the `test` subset, but it is important to avoid it when tuning the methods.

In [13]:
classifier = LogisticRegression(solver='lbfgs', max_iter=1000)
classifier.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

Let's check the performance on the training data:

In [14]:
print('Features results: train')
pred_train = classifier.predict(X_train)
print(accuracy_score(y_train, pred_train))

Features results: train
0.78825


... and the validation dataset:

In [15]:
print('Features results: validation')
pred_valid = classifier.predict(X_valid)
print(accuracy_score(y_valid, pred_valid))

Features results: validation
0.7872


<a name='e3'></a>
#### Exercise 3: Improving the features

(5p) Iteratively improve your hand-crafted features. Think about what information from the review might be useful for to predict the rating a person gave to the particular movie. You can also look into the expected format (or range) of features for the classifier.

Document the steps you tried (even if unsuccessful) and how they influenced the metrics. Try at least 3 modifications from your original implementation.

In [16]:
### YOUR CODE HERE
def _tokens(text):
    return _word_re.findall((text or "").lower())

def features_v0(text):
    return calculate_features(text)

def features_v1(text):
    # Add casing emphasis + average word length
    t = text or ""
    toks = _tokens(t)
    base = features_v0(t).tolist()
    num_tokens = len(toks)
    avg_wlen = (sum(len(w) for w in toks) / num_tokens) if num_tokens else 0.0
    caps_ratio = (sum(1 for ch in t if ch.isupper()) / max(1, sum(1 for ch in t if ch.isalpha())))
    base.extend([avg_wlen, caps_ratio])
    return np.array(base, dtype=float)

def features_v2(text):
    # Add “intensity” and “elongation” + quotes
    t = text or ""
    toks = _tokens(t)
    base = features_v0(t).tolist()
    elongated = len(re.findall(r"(.)\1\1+", t.lower()))  # e.g., "soooo"
    quotes = t.count('"') + t.count("'")
    base.extend([float(elongated), float(quotes)])
    return np.array(base, dtype=float)

def features_v3(text):
    # Add simple negation flip patterns like "not good", "not bad"
    t = text or ""
    toks = _tokens(t)
    base = features_v0(t).tolist()
    bigrams = list(zip(toks, toks[1:]))
    not_pos = sum(1 for a,b in bigrams if a in _NEGATORS and b in POSITIVE_WORDS)
    not_neg = sum(1 for a,b in bigrams if a in _NEGATORS and b in NEGATIVE_WORDS)
    base.extend([float(not_pos), float(not_neg)])
    return np.array(base, dtype=float)

feature_versions = [
    ("v0_base", features_v0),
    ("v1_caps+avglen", features_v1),
    ("v2_elong+quotes", features_v2),
    ("v3_negation_bigrams", features_v3),
]

results = []
for name, fn in feature_versions:
    Xt = calculate_features_dataset(dataset["train"], fn)
    Xv = calculate_features_dataset(dataset["validation"], fn)

    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(solver="lbfgs", max_iter=2000))
    ])
    clf.fit(Xt, y_train)

    pred_v = clf.predict(Xv)
    acc = accuracy_score(y_valid, pred_v)
    # f1 will be defined in Ex4 cell as PRIMARY_METRIC; if not yet, fallback
    try:
        from sklearn.metrics import f1_score
        f1 = f1_score(y_valid, pred_v)
    except Exception:
        f1 = float("nan")

    results.append((name, acc, f1))

print("Feature iteration results (validation):")
for name, acc, f1 in results:
    print(f"{name:22s}  acc={acc:.4f}  f1={f1:.4f}")

### YOUR CODE ENDS HERE

Extracting features: 100%|██████████| 5000/5000 [00:01<00:00, 3042.08it/s]


Feature iteration results (validation):
v0_base                 acc=0.7856  f1=0.7918
v1_caps+avglen          acc=0.7866  f1=0.7928
v2_elong+quotes         acc=0.7878  f1=0.7943
v3_negation_bigrams     acc=0.7860  f1=0.7924


- **Baseline (v0_base):**
  - log(#chars)
  - log(#tokens)
  - number of unique punctuation characters
  - counts of positive/negative lexicon words
  - polarity (pos − neg)
  - counts of `!` and `?`
  - negation count
  - normalized pos/neg ratios

- **Modification 1 (v1_caps+avglen):**
  - Added average token length
  - Added ratio of uppercase letters
  - **Interpretation:** capitalization in IMDB reviews is noisy (titles, names) and doesn’t reliably indicate sentiment

- **Modification 2 (v2_elong+quotes):**
  - Added count of elongated character sequences
  - Added quote count
  - **Interpretation:** elongation/quotes are rare and inconsistent; adds noise more than signal

- **Modification 3 (v3_negation_bigrams):**
  - Added counts of patterns like “not good” and “not bad” by checking negator + sentiment word bigrams
  - **Interpretation:** negation handling fixes a real weakness of lexicon-counting features, because it captures polarity flips

<a name='e4'></a>
#### Exercise 4: Improving the evaluation

(5p) In the previous cells, we only looked at the accuracy of predictions. Investigate which other metrics might be better for our case. You can check the documentation of scikit-learn for evaluation metrics ([https://scikit-learn.org/stable/api/sklearn.metrics.html#classification-metrics](https://scikit-learn.org/stable/api/sklearn.metrics.html#classification-metrics)). Give reasons why the metrics you try can be more informative than raw accuracy score.

Decide which evaluation metric(s) is most suitable for our use-case and give reasons why. Test your features-based classifier and all further classifiers on that metric (apart from the accuracy score).

In [17]:
### YOUR CODE HERE
PRIMARY_METRIC = "f1"  # sensible for binary sentiment; IMDB is roughly balanced

def evaluate_classifier(clf, X, y, name="model"):
    pred = clf.predict(X)

    metrics = {
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
    }

    score = None
    if hasattr(clf, "predict_proba"):
        try:
            score = clf.predict_proba(X)[:, 1]
        except Exception:
            score = None
    elif hasattr(clf, "decision_function"):
        try:
            score = clf.decision_function(X)
        except Exception:
            score = None

    if score is not None:
        try:
            metrics["roc_auc"] = roc_auc_score(y, score)
        except Exception:
            pass

    print(f"\n[{name}]")
    for k, v in metrics.items():
        print(f"{k:10s}: {v:.4f}")
    print("confusion_matrix:\n", confusion_matrix(y, pred))
    return metrics

# Evaluate the already-trained features classifier (the one stored in `classifier`)
features_classifier = classifier
_ = evaluate_classifier(features_classifier, X_train, y_train, name="Features (train)")
_ = evaluate_classifier(features_classifier, X_valid, y_valid, name="Features (validation)")

### YOUR CODE ENDS HERE


[Features (train)]
accuracy  : 0.7883
precision : 0.7711
recall    : 0.8201
f1        : 0.7948
roc_auc   : 0.8691
confusion_matrix:
 [[7562 2435]
 [1800 8203]]

[Features (validation)]
accuracy  : 0.7872
precision : 0.7699
recall    : 0.8186
f1        : 0.7935
roc_auc   : 0.8628
confusion_matrix:
 [[1892  611]
 [ 453 2044]]


- Precision (for positive class): Of the reviews predicted positive, how many are actually positive?

- Recall (for positive class): Of the truly positive reviews, how many did we catch?

- F1 score: Harmonic mean of precision and recall. This is more informative than accuracy when errors are asymmetric or when we want a balance between false positives and false negatives.

- ROC-AUC: Measures ranking quality across thresholds.

- Confusion matrix: Shows error types

- Chosen primary metric: F1. Even though IMDB is roughly balanced, F1 still prevents “accuracy optimism” when precision/recall trade off.

- Results shows recall > precision: the model catches positives fairly well but produces a noticeable number of false positives

## 3. Bag-of-Words Classifier

Similar to the previous lab, we will use the classic bag-of-words representation as one of our embeddings. While it is simple and does not preserve the positions of words, it gives our classifier a lot of useful information.

<a name='e5'></a>
#### Exercise 5: Implementing BOW

(5p) Implement the BOW. In this exercise, we do not give you a rigid structure, so you can conjure your own. The two things your code should produce is the `token_to_id` dictionary, and `bag_of_words()` function that accepts a list of tokens, and the `token_to_id` dictionary while generating the BOW representation as a numpy array.

In [18]:
#### YOUR CODE HERE

MAX_VOCAB_SIZE = 1_000

# The goal is to implement the `bag_of_words(tokens, token_to_id)` function similar to the previous lab.
# You might want to follow the steps:
# - tokenize the `text` column in the dataset,
# - extract the vocabulary from the tokens,
# - limit the vocabulary to `MAX_VOCAB_SIZE`,
# - calculate the `token_to_id` dictionary
# - implement the `bag_of_words(tokens, token_to_id)` function.
def simple_tokenize(text: str):
    # Lowercased word tokens for classic BOW
    return re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+", (text or "").lower())

def add_tokens(example):
    example["tokens"] = simple_tokenize(example["text"])
    return example

# Ensure the dataset has tokens for all splits
dataset = dataset.map(add_tokens, desc="Tokenizing for BOW")

# Build vocabulary from TRAIN only (no leakage!)
token_counts = Counter()
for ex in dataset["train"]:
    token_counts.update(ex["tokens"])

token_to_id = {}
for tok, _ in token_counts.most_common(MAX_VOCAB_SIZE - 1):
    token_to_id[tok] = len(token_to_id)

def bag_of_words(tokens, token_to_id):
    """
    Creates a bag-of-words representation of the sentence
    Args:
        tokens: a list of tokens
        token_to_id: a dictionary mapping each word to an index in the vocabulary

    Returns:: a numpy array of size vocab_size with the counts of each word in the vocabulary
    """
    vocab_size = len(token_to_id)
    bow = np.zeros(vocab_size, dtype=float)
    unk_id = token_to_id.get("<UNK>", None)

    for t in tokens:
        idx = token_to_id.get(t, unk_id)
        if idx is None:
            continue
        bow[idx] += 1.0
    return bow
#### YOUR CODE ENDS HERE

Tokenizing for BOW:   0%|          | 0/20000 [00:00<?, ? examples/s]

Tokenizing for BOW:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing for BOW:   0%|          | 0/25000 [00:00<?, ? examples/s]

Here, we will use your implemented function to calculate the bag-of-words for each example in the `train` and `validation` subsets.



train_bows = []
for example in tqdm.tqdm(dataset['train'], desc='Calculating test BOWs'):
    train_bows.append(bag_of_words(example['tokens'], token_to_id))
train_bows = np.array(train_bows, dtype=float)

valid_bows = []
for example in tqdm.tqdm(dataset['validation'], desc='Calculating validation BOWs'):
    valid_bows.append(bag_of_words(example['tokens'], token_to_id))
valid_bows = np.array(valid_bows, dtype=float)

Finally, we can train the classifier on the BOW representations and the labels in the `train` split.

In [19]:
train_bows = []
for example in dataset['train']:
    train_bows.append(bag_of_words(example['tokens'], token_to_id))

valid_bows = []
for example in dataset['validation']:
    valid_bows.append(bag_of_words(example['tokens'], token_to_id))

In [20]:
classifier = LogisticRegression(solver='lbfgs', max_iter=1000)
print('Training classifier...')
classifier.fit(train_bows, y_train)

Training classifier...


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

Let's evaluate the classifier:

In [21]:
print('BOW results: train')
pred_train = classifier.predict(train_bows)
print(accuracy_score(y_train, pred_train))

print('BOW results: validation')
pred_valid = classifier.predict(valid_bows)
print(accuracy_score(y_valid, pred_valid))

BOW results: train
0.88665
BOW results: validation
0.8548


<a name='e6'></a>
#### Exercise 6: Tuning the model

(5p) Try different values for the vocab size. Experiment with adding the hand-crafted features. Test the model on the evaluation metric of your choice (remember to use the validation split).

In [22]:
#### YOUR CODE HERE
def build_token_to_id(train_split, max_vocab_size, min_freq=2):
    counts = Counter()
    for ex in train_split:
        counts.update(ex["tokens"])
    # filter rare tokens a bit (helps reduce noise)
    items = [(t, c) for t, c in counts.items() if c >= min_freq]
    items.sort(key=lambda x: x[1], reverse=True)

    t2i = {"<UNK>": 0}
    for tok, _ in items[: max_vocab_size - 1]:
        t2i[tok] = len(t2i)
    return t2i

def bows_for_split(split, t2i, desc="BoW"):
    V = len(t2i)
    unk = t2i.get("<UNK>", 0)
    X = np.zeros((len(split), V), dtype=float)
    for i, ex in enumerate(tqdm.tqdm(split, desc=desc)):
        for tok in ex["tokens"]:
            X[i, t2i.get(tok, unk)] += 1.0
    return X

vocab_sizes = [500, 1000, 2000, 5000]
use_handcrafted_options = [False, True]

best = None  # (metric, vocab_size, use_handcrafted, clf, t2i, Xtr, Xva)

for vs in vocab_sizes:
    t2i = build_token_to_id(dataset["train"], vs, min_freq=2)
    Xtr_bow = bows_for_split(dataset["train"], t2i, desc=f"BoW train (V={vs})")
    Xva_bow = bows_for_split(dataset["validation"], t2i, desc=f"BoW valid (V={vs})")

    for use_hc in use_handcrafted_options:
        if use_hc:
            Xtr = np.concatenate([Xtr_bow, X_train], axis=1)
            Xva = np.concatenate([Xva_bow, X_valid], axis=1)
            name = f"BoW+HC (V={vs})"
        else:
            Xtr, Xva = Xtr_bow, Xva_bow
            name = f"BoW (V={vs})"

        clf = LogisticRegression(solver="lbfgs", max_iter=2000)
        clf.fit(Xtr, y_train)
        metrics = evaluate_classifier(clf, Xva, y_valid, name=name)

        score = metrics.get(PRIMARY_METRIC, metrics["accuracy"])
        if (best is None) or (score > best[0]):
            best = (score, vs, use_hc, clf, t2i, Xtr_bow, Xva_bow)

print("\nBest BoW setting (by PRIMARY_METRIC):")
print(f"score={best[0]:.4f}  vocab_size={best[1]}  use_handcrafted={best[2]}")

# Update globals to reflect best pure BoW vocabulary (so later test eval uses it)
_best_score, _best_vs, _best_use_hc, _best_clf, _best_t2i, _best_train_bow, _best_valid_bow = best

token_to_id = _best_t2i
train_bows = _best_train_bow
valid_bows = _best_valid_bow

# Store best classifier and whether it used handcrafted features
bow_classifier = _best_clf
bow_uses_handcrafted = bool(_best_use_hc)
#### YOUR CODE ENDS HERE

BoW valid (V=500): 100%|██████████| 5000/5000 [00:01<00:00, 2611.65it/s]



[BoW (V=500)]
accuracy  : 0.8326
precision : 0.8265
recall    : 0.8414
f1        : 0.8339
roc_auc   : 0.9097
confusion_matrix:
 [[2062  441]
 [ 396 2101]]

[BoW+HC (V=500)]
accuracy  : 0.8522
precision : 0.8423
recall    : 0.8662
f1        : 0.8541
roc_auc   : 0.9232
confusion_matrix:
 [[2098  405]
 [ 334 2163]]


BoW valid (V=1000): 100%|██████████| 5000/5000 [00:01<00:00, 2693.92it/s]



[BoW (V=1000)]
accuracy  : 0.8552
precision : 0.8472
recall    : 0.8662
f1        : 0.8566
roc_auc   : 0.9267
confusion_matrix:
 [[2113  390]
 [ 334 2163]]

[BoW+HC (V=1000)]
accuracy  : 0.8572
precision : 0.8503
recall    : 0.8666
f1        : 0.8584
roc_auc   : 0.9299
confusion_matrix:
 [[2122  381]
 [ 333 2164]]


BoW valid (V=2000): 100%|██████████| 5000/5000 [00:01<00:00, 2679.94it/s]



[BoW (V=2000)]
accuracy  : 0.8666
precision : 0.8672
recall    : 0.8654
f1        : 0.8663
roc_auc   : 0.9332
confusion_matrix:
 [[2172  331]
 [ 336 2161]]

[BoW+HC (V=2000)]
accuracy  : 0.8664
precision : 0.8689
recall    : 0.8626
f1        : 0.8658
roc_auc   : 0.9331
confusion_matrix:
 [[2178  325]
 [ 343 2154]]


BoW valid (V=5000): 100%|██████████| 5000/5000 [00:02<00:00, 2272.85it/s]



[BoW (V=5000)]
accuracy  : 0.8638
precision : 0.8612
recall    : 0.8670
f1        : 0.8641
roc_auc   : 0.9318
confusion_matrix:
 [[2154  349]
 [ 332 2165]]

[BoW+HC (V=5000)]
accuracy  : 0.8652
precision : 0.8630
recall    : 0.8678
f1        : 0.8654
roc_auc   : 0.9314
confusion_matrix:
 [[2159  344]
 [ 330 2167]]

Best BoW setting (by PRIMARY_METRIC):
score=0.8663  vocab_size=2000  use_handcrafted=False


## 4. BERT Model

For the first part of this lab, we will be using a pre-trained BERT model from Huggingface, namely the [BERT Cased](https://huggingface.co/google-bert/bert-base-cased). You can read the original paper that introduced this model [here](https://aclanthology.org/N19-1423.pdf). This paper has been one of the most cited papers ever (currently having more than 100,000 citations).

We will specify the model name that can be found on the model's card on huggingface (revisit the first link). Make sure to check what other information Huggingface is offering (e.g. how to use the model, limitations, how to inference, etc.).

In [23]:
model_name = 'google-bert/bert-base-cased'

### 4.1 Tokenizer

The models on huggingface come with their own tokenizers. They are loaded separately from the models. We can use [AutoTokenizer](https://huggingface.co/docs/transformers/v4.40.2/en/model_doc/auto#transformers.AutoTokenizer)'s `from_pretrained()` method to load it.

Inspect the output: The loaded object is of `BertTokenizer` class. Check the documentation [here](https://huggingface.co/docs/transformers/en/model_doc/bert#transformers.BertTokenizer).

In [24]:
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
print(tokenizer)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BertTokenizer(name_or_path='google-bert/bert-base-cased', vocab_size=28996, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


Next, let's see how we can use it to tokenize some text.

In [25]:
print(dataset['test'][0]['text'])
tokenized = tokenizer(dataset['test'][0]['text'], padding=True, return_tensors='pt')
print("---")
print(type(tokenized))
print("---")
print(tokenized)

I saw (unfortunately) the dubbed version on Encore. Student Paula Henning (Franka Potente who was also in the cult favorite "Run Lola Run") stars as a serious medical student who gets into a prestigious school in Germany. But she soon discovers that some students go missing and the bodies they work on in the anatomy lab are incredibly fresh... I was stuck seeing the dubbed version on Encore. It hurt a lot (the words not matching the lips got annoying real quick) but I still liked what I saw. The acting was good, it was beautifully photographed, it wasn't TOO gruesome and I was never bored. Even more refreshing was a likable heroine who fights back when the bad guys go after her. The (mild) nudity was, in a refreshing twist, male! A previous poster mentions Benno Furmann (who is excellent) showed his butt but I don't remember seeing it. Regardless this is a well done, scary and excellent thriller. From all I've read the original German language version is the best (I don't doubt that fo

Examine the outputs: The tokenizer returned three things:
- `input_ids` - this is a PyTorch tensor ([https://pytorch.org/docs/stable/tensors.html](https://pytorch.org/docs/stable/tensors.html)) with the indices of our tokens. PyTorch tensors are similar to numpy arrays. They hold data in a multidimensional array or matrix. The difference is that PyTorch tensors can be placed and modified on the GPU which greatly improves the speed of execution.
- `token_type_ids` - this tensor holds the information about the index of the sentence. This has to do with the classification objective from the original paper, where two sentences were given and the model had to predict if they are connected. Because we only included a single sentence, we have only zeros here. We will not be concerned with it in this lab.
- `attention_mask` - holds the mask that the model will use to determine if the tokens in the `input_ids` are the real tokens or *padding*. Padding is a technique used to ensure that all input sequences have the same length. BERT (like many other NLP models) process data in batches and requires each sequence in a batch to have the same length, so sequences that are shorter than the maximum sequence length in the batch are padded with special tokens. In this case, because we only inputted a single sentence, the mask contains only ones. Later you will see examples where this is not the case.

Let's see how exactly the sentence was tokenized and how we can retrieve the original text. Notice that some words have been split into multiple tokens (remember when we discussed sub-word tokenization in class?). Also pay attention to the added special tokens, namely `CLS` and `SEP`:

The `[CLS]` token is a special classification token added at the beginning of every input sequence. It stands for "classification" (daah!) and is used by BERT to aggregate information from the entire sequence. The final hidden state corresponding to this token (after passing through the transformer layers) is used as the aggregate sequence representation for classification tasks. We will use this later in the lab!

The `[SEP]` token is used to separate different segments or sentences within the input sequence. It stands for "separator" (daaah again!).

In [26]:
print(tokenized['input_ids'].shape)
print("---")
print(tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0]))
print("---")
print(len(tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0])))
print("---")
print(tokenizer.decode(tokenized['input_ids'][0]))
print("---")
print(tokenizer.decode(tokenized['input_ids'][0], skip_special_tokens=True))

torch.Size([1, 254])
---
['[CLS]', 'I', 'saw', '(', 'unfortunately', ')', 'the', 'dubbed', 'version', 'on', 'En', '##core', '.', 'Student', 'Paula', 'He', '##nning', '(', 'Frank', '##a', 'Po', '##ten', '##te', 'who', 'was', 'also', 'in', 'the', 'cult', 'favorite', '"', 'Run', 'Lola', 'Run', '"', ')', 'stars', 'as', 'a', 'serious', 'medical', 'student', 'who', 'gets', 'into', 'a', 'prestigious', 'school', 'in', 'Germany', '.', 'But', 'she', 'soon', 'discovers', 'that', 'some', 'students', 'go', 'missing', 'and', 'the', 'bodies', 'they', 'work', 'on', 'in', 'the', 'anatomy', 'lab', 'are', 'incredibly', 'fresh', '.', '.', '.', 'I', 'was', 'stuck', 'seeing', 'the', 'dubbed', 'version', 'on', 'En', '##core', '.', 'It', 'hurt', 'a', 'lot', '(', 'the', 'words', 'not', 'matching', 'the', 'lips', 'got', 'annoying', 'real', 'quick', ')', 'but', 'I', 'still', 'liked', 'what', 'I', 'saw', '.', 'The', 'acting', 'was', 'good', ',', 'it', 'was', 'beautifully', 'photographed', ',', 'it', 'wasn', "'", 

Tokenizer can process a list of sentences. This will create a batched output with tensor's first dimension corresponding to the batch size (the number of sentences we passed to the tokenizer). Examine the following cell and make sure it makes sense to you.

In [27]:
print(dataset['test'][0:3]['text'])
tokenized = tokenizer(dataset['test'][0:3]['text'], padding=True, return_tensors='pt')
print(tokenized)
print(tokenized['input_ids'].shape)
print(tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0]))
print(len(tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0])))
print(tokenizer.decode(tokenized['input_ids'][0]))
print(tokenizer.decode(tokenized['input_ids'][0], skip_special_tokens=True))

['I saw (unfortunately) the dubbed version on Encore. Student Paula Henning (Franka Potente who was also in the cult favorite "Run Lola Run") stars as a serious medical student who gets into a prestigious school in Germany. But she soon discovers that some students go missing and the bodies they work on in the anatomy lab are incredibly fresh... I was stuck seeing the dubbed version on Encore. It hurt a lot (the words not matching the lips got annoying real quick) but I still liked what I saw. The acting was good, it was beautifully photographed, it wasn\'t TOO gruesome and I was never bored. Even more refreshing was a likable heroine who fights back when the bad guys go after her. The (mild) nudity was, in a refreshing twist, male! A previous poster mentions Benno Furmann (who is excellent) showed his butt but I don\'t remember seeing it. Regardless this is a well done, scary and excellent thriller. From all I\'ve read the original German language version is the best (I don\'t doubt t

<a name='e7'></a>
#### Exercise 7: Questions about the tokenizer

Answer the following questions:
- (1p) What is the size of the vocabulary?
- (2p) What are the special tokens apart from `[CLS]` and `[SEP]`? What are their functions?

- Vocabulary size: 28,996 tokens
- Special tokens besides [CLS] and [SEP]:

  - [PAD]: padding token used to make sequences in a batch the same length (attention mask marks padding so the model can ignore it).

  - [UNK]: unknown token used when a string cannot be represented by known subword pieces.

  - [MASK]: used during BERT’s masked language modeling (MLM) pretraining objective to hide tokens and predict them

- Additionally, BERT uses segment information (via token_type_ids) for sentence-pair tasks: tokens are marked as belonging to sentence A or sentence B (this corresponds to the “segment embeddings” used in NSP-style setups)

### 4.2 Loading the Model

In this section, we will load and examine the model. We will start with selecting the device we will place the model on. This will be a GPU (if one is available) or a CPU.

Google Colab offers free access to GPU, provided there is availability (also based on quotas which may vary based on your usage and the overall demand on Colab's resources). If you are working locally, then if you don't have a GPU, CPU will be selected. For the first parts of the assignment running on CPU might be okay but when we have to process the dataset a GPU will be necessary.

The following cell will select the device for us.

In [28]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda:0


Now, let's load the model from huggingface and move it (slowly because it's heavy due to the large number of parameters) on the device from the previous cell (the methods `to()`).

In [29]:
model = transformers.AutoModel.from_pretrained(model_name)
print('loaded on device:', model.device)
model.to(device)
print('moved to device', model.device)
print(model)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loaded on device: cpu
moved to device cuda:0
BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(28996, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          

When loading the model you might have seen the warning about some unexpected weights. This means that the model on huggingface has some additional weights that were downloaded, but our model does not use them. In essence, you can load the same weights (as linked by our `model_name`) to load to different but related models. In our case those would be `BertForMaskedLM` or `BertForNextSentencePrediction` instead of our `BertModel`, which is loaded automatically as the `AutoModel`. Below is a way to load the weights into a different model.

In [30]:
transformers.BertForMaskedLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: google-bert/bert-base-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

Next, let's use BERT model for inference. We will tokenize the first sentence of our dataset and pass it to the model. We set `output_hidden_states` to `True` in order to have access to the hidden states of the model. Those represent the latent representations after embedding and transformer layers.

In [31]:
tokenized = tokenizer(dataset['test'][0]['text'], padding=True, return_tensors='pt').to(device)
print(tokenized)
model_output = model(**tokenized, output_hidden_states=True)

{'input_ids': tensor([[  101,   146,  1486,   113, 16679,   114,  1103,  9098,  1683,  1113,
         13832,  9475,   119,  7646, 14153,  1124, 15918,   113,  2748,  1161,
         18959,  5208,  1566,  1150,  1108,  1145,  1107,  1103,  9528,  5095,
           107,  6728, 15976,  6728,   107,   114,  2940,  1112,   170,  3021,
          2657,  2377,  1150,  3370,  1154,   170,  8593,  1278,  1107,  1860,
           119,  1252,  1131,  1770,  9149,  1115,  1199,  1651,  1301,  3764,
          1105,  1103,  3470,  1152,  1250,  1113,  1107,  1103, 19768,  8074,
          1132, 12170,  4489,   119,   119,   119,   146,  1108,  5342,  3195,
          1103,  9098,  1683,  1113, 13832,  9475,   119,  1135,  2644,   170,
          1974,   113,  1103,  1734,  1136,  9901,  1103,  2089,  1400, 17090,
          1842,  3613,   114,  1133,   146,  1253,  3851,  1184,   146,  1486,
           119,  1109,  3176,  1108,  1363,   117,  1122,  1108, 19758, 17411,
           117,  1122,  1445,   112,  

Examine the next cell and make sure everything makes sense to you. Consult the [documentation](https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertModel.forward) in case of doubt.

In [32]:
print(list(model_output.keys()))
print()
print('pooler_output:')
print(type(model_output['pooler_output']))
print(model_output['pooler_output'].shape)
print()
print('hidden_states:')
print(type(model_output['hidden_states']))
print(len(model_output['hidden_states']))
print(type(model_output['hidden_states'][0]))
print(model_output['hidden_states'][0].shape)
print()
print('last_hidden_state:')
print(type(model_output['last_hidden_state']))
print(model_output['last_hidden_state'].shape)

['last_hidden_state', 'pooler_output', 'hidden_states']

pooler_output:
<class 'torch.Tensor'>
torch.Size([1, 768])

hidden_states:
<class 'tuple'>
13
<class 'torch.Tensor'>
torch.Size([1, 254, 768])

last_hidden_state:
<class 'torch.Tensor'>
torch.Size([1, 254, 768])


<a name='e8'></a>
#### Exercise 8: Questions about the Model

Examine the output of the previous cells. Answer the following questions:
- (1p) What is the number of transformer layers in this model?
- (1p) What is the dimension of the embeddings?
- (1p) What is the hidden size of the FFN in the transformer layer?
- (1p) What is the total number of parameters of the model (hint: check the `num_parameters()` method of the model)?
- (1p) How can you find the vocabulary size from the model?
- (1p) What is the length of the `hidden_states` in the output? Why?

- Number of transformer layers: 12 (BERT-base encoder stack)
- Embedding dimension (hidden size): 768
- Hidden size of the FFN (intermediate size): 3072 (the feed-forward sublayer expands 768 → 3072 → 768 in each transformer block)
- Total number of parameters: about ~100M+ (≈108M) parameters for BERT-base
- How to find vocabulary size from the model: model.config.vocab_size (or model.get_input_embeddings().num_embeddings)
- Length of hidden_states and why: hidden_states has length 13 because it includes:

  - the output of the embedding layer (layer 0),

  - plus the outputs after each of the 12 transformer layers (layers 1–12),
  when output_hidden_states=True

## 5. BERT Sentence Embeddings

Having the model loaded and ready we can work on obtaining the sentence embeddings. During the last lab, you averaged the token embeddings. This time we will start with something else. Remember the CLS token? Its hidden representation is often used for classification as a representation of the whole sentence. We will do exactly that.

But first, we have to tokenize the dataset using BERT tokenizer.

<a name='e9'></a>
#### Exercise 9: BERT tokenizing examples

(5p) Fill in the following function to embed the examples (passed as a parameter) using the tokenizer (also a parameter). The function will tokenize a batch of examples, but the tokenizer can handle that, if you remember from the previous section.

In [33]:
def tokenize_text_bert(examples, tokenizer):
    """
    Tokenizes the `sentence` column from the batch of examples and returns the whole output of the tokenizer.
    Args:
        examples: a batch of examples
        tokenizer: the BERT tokenizer

    Returns: the tokenized `sentence` column (returns the whole output of the tokenizer)

    """
    ### YOUR CODE HERE
    # Tokenize a batch of texts. Don't return tensors here (HF datasets expects lists).
    tokenized_sentence = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding=False,
    )
    ### YOUR CODE ENDS HERE
    return tokenized_sentence


In [34]:
dataset_tokenized_bert = dataset.map(tokenize_text_bert,
                                     fn_kwargs={'tokenizer': tokenizer},
                                     batched=True,
                                     remove_columns=dataset['train'].column_names,)
print(dataset_tokenized_bert)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
})


<a name='e10'></a>
#### Exercise 10: BERT sentence embeddings by the CLS token

(5p) Implement the following function which calculates the sentence embeddings based on the model output (passed to the function as a parameter). It should take the embedding of the CLS token of last layer.

In [35]:
def calculate_cls_embeddings(input_batch, model_output):
    """
    Calculates the sentence embeddings of a batch of sentences as the last-layer representation of the CLS token.
    Args:
        input_batch: tokenized batch of sentences (as returned by the tokenizer), contains `input_ids`, `token_type_ids`, and `attention_mask` tensors
        model_output: the output of the model given the `input_batch`, contains `last_hidden_state`, `pooler_output`, `hidden_states` tensors

    Returns: tensor of the hidden states of the CLS token (from the last layer) for each example in the batch

    """

    ### YOUR CODE HERE
    last_hidden = model_output["last_hidden_state"]  # [B, T, H]
    sentence_embeddings = last_hidden[:, 0, :]        # [B, H]
    ### YOUR CODE ENDS HERE

    return sentence_embeddings

In [36]:
text = "The weather is nice today."
tokenized = tokenizer(text, padding=True, return_tensors='pt').to(device)
print(tokenized)
model_output = model(**tokenized, output_hidden_states=True)
print(model_output['last_hidden_state'].shape)
sentence_embedding = calculate_cls_embeddings(tokenized, model_output)
print(sentence_embedding.shape)

{'input_ids': tensor([[ 101, 1109, 4250, 1110, 3505, 2052,  119,  102]], device='cuda:0'), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}
torch.Size([1, 8, 768])
torch.Size([1, 768])


In [37]:
def embed_dataset(dataset, model, sentence_embedding_fn, batch_size=8):
    data_collator = transformers.DataCollatorWithPadding(tokenizer)
    data_loader = DataLoader(dataset, batch_size=batch_size, collate_fn=data_collator)
    sentence_embeddings = []
    with torch.no_grad():
        for batch in tqdm.tqdm(data_loader):
            batch.to(device)
            model_output = model(**batch, output_hidden_states=True)
            batch_sentence_embeddings = sentence_embedding_fn(batch, model_output)
            sentence_embeddings.append(batch_sentence_embeddings.detach().cpu())

    sentence_embeddings = torch.concat(sentence_embeddings, dim=0)
    return sentence_embeddings

In [38]:
bert_cls_train = embed_dataset(dataset_tokenized_bert['train'], model, calculate_cls_embeddings)
print(bert_cls_train.shape)

bert_cls_valid = embed_dataset(dataset_tokenized_bert['validation'], model, calculate_cls_embeddings)
print(bert_cls_valid.shape)

100%|██████████| 2500/2500 [10:35<00:00,  3.93it/s]


torch.Size([20000, 768])


100%|██████████| 625/625 [02:43<00:00,  3.82it/s]

torch.Size([5000, 768])


In [39]:
classifier = LogisticRegression(solver='lbfgs', max_iter=1000)
print('Training classifier...')
classifier.fit(bert_cls_train, y_train)

Training classifier...


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [40]:
print('BERT train')
pred_train = classifier.predict(bert_cls_train)
print(accuracy_score(y_train, pred_train))

print('BERT valid')
pred_valid = classifier.predict(bert_cls_valid)
print(accuracy_score(y_valid, pred_valid))

BERT train
0.86465
BERT valid
0.8454


You can test the model on the evaluation metric of your choice:

In [41]:
#### YOUR CODE HERE
bert_cls_classifier = classifier
_ = evaluate_classifier(bert_cls_classifier, bert_cls_train, y_train, name="BERT-CLS (train)")
_ = evaluate_classifier(bert_cls_classifier, bert_cls_valid, y_valid, name="BERT-CLS (valid)")
#### YOUR CODE ENDS HERE


[BERT-CLS (train)]
accuracy  : 0.8647
precision : 0.8701
recall    : 0.8574
f1        : 0.8637
roc_auc   : 0.9383
confusion_matrix:
 [[8716 1281]
 [1426 8577]]

[BERT-CLS (valid)]
accuracy  : 0.8454
precision : 0.8530
recall    : 0.8342
f1        : 0.8435
roc_auc   : 0.9212
confusion_matrix:
 [[2144  359]
 [ 414 2083]]


<a name='e11'></a>
#### Exercise 11: BERT Sentence embeddings by averaging tokens

(5p) Implement embedding sentences by averaging the hidden representations of tokens. Make sure to ignore the special and padding tokens. The padding tokens are indicated by the attention mask. You can find the other special tokens using the tokenizer's attributes such as `tokenizer.sep_token_id`. The function accepts the `layer` parameter. Typically, you would use the hidden representations of the last layer, it might be beneficial for some tasks to use previous layers or an averaged representations of multiple layers.

In [42]:
### YOUR CODE HERE

def calculate_sentence_embeddings(input_batch, model_output, layer=-1):
    """
    Calculates the sentence embeddings of a batch of sentences as a mean of token representations.
    The representations are taken from the layer of the index provided as a `layer` parameter.
    Args:
        input_batch: tokenized batch of sentences (as returned by the tokenizer), contains `input_ids`, `token_type_ids`, and `attention_mask` tensors
        model_output: the output of the model given the `input_batch`, contains `last_hidden_state`, `pooler_output`, `hidden_states` tensors
        layer: specifies the layer of the hidden states that are used to calculate sentence embedding

    Returns: tensor of the averaged hidden states (from the specified layer) for each example in the batch

    """
    attention_mask = input_batch['attention_mask']
    hidden_states = model_output['hidden_states'][layer]

    ### YOUR CODE HERE
    input_ids = input_batch["input_ids"]            # [B, T]
    attn = attention_mask.bool()                    # True for non-padding

    # Exclude special tokens (CLS, SEP, PAD)
    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
    pad_id = tokenizer.pad_token_id

    non_special = attn
    if cls_id is not None:
        non_special = non_special & (input_ids != cls_id)
    if sep_id is not None:
        non_special = non_special & (input_ids != sep_id)
    if pad_id is not None:
        non_special = non_special & (input_ids != pad_id)

    mask = non_special.unsqueeze(-1).to(hidden_states.dtype)  # [B, T, 1]

    summed = (hidden_states * mask).sum(dim=1)                # [B, H]
    counts = mask.sum(dim=1).clamp(min=1.0)                   # [B, 1]
    sentence_embeddings = summed / counts                     # [B, H]
    ### YOUR CODE ENDS HERE

    return sentence_embeddings

### YOUR CODE ENDS HERE

We can test it here:


In [43]:
text = "The weather is nice today."
tokenized = tokenizer(text, padding=True, return_tensors='pt').to(device)
print(tokenized)
model_output = model(**tokenized, output_hidden_states=True)
print(model_output['last_hidden_state'].shape)
sentence_embedding = calculate_sentence_embeddings(tokenized, model_output)
print(sentence_embedding.shape)

{'input_ids': tensor([[ 101, 1109, 4250, 1110, 3505, 2052,  119,  102]], device='cuda:0'), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}
torch.Size([1, 8, 768])
torch.Size([1, 768])


We will embed the sentences and evaluate the model on the `validation` subset.


In [44]:
bert_sentence_train = embed_dataset(dataset_tokenized_bert['train'], model, calculate_sentence_embeddings)
print(bert_cls_train.shape)

bert_sentence_valid = embed_dataset(dataset_tokenized_bert['validation'], model, calculate_sentence_embeddings)
print(bert_cls_valid.shape)

100%|██████████| 2500/2500 [10:44<00:00,  3.88it/s]


torch.Size([20000, 768])


100%|██████████| 625/625 [02:45<00:00,  3.77it/s]

torch.Size([5000, 768])


In [45]:
classifier = LogisticRegression(solver='lbfgs', max_iter=1000)
print('Training classifier...')
classifier.fit(bert_sentence_train, y_train)

Training classifier...


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [46]:
print('BERT train')
pred_train = classifier.predict(bert_sentence_train)
print(accuracy_score(y_train, pred_train))

print('BERT valid')
pred_valid = classifier.predict(bert_sentence_valid)
print(accuracy_score(y_valid, pred_valid))

BERT train
0.8985
BERT valid
0.88


Test the model on the evaluation metric of your choice:


In [47]:
#### YOUR CODE HERE
bert_mean_classifier = classifier
_ = evaluate_classifier(bert_mean_classifier, bert_sentence_train, y_train, name="BERT-Mean (train)")
_ = evaluate_classifier(bert_mean_classifier, bert_sentence_valid, y_valid, name="BERT-Mean (valid)")
#### YOUR CODE ENDS HERE


[BERT-Mean (train)]
accuracy  : 0.8985
precision : 0.9014
recall    : 0.8949
f1        : 0.8982
roc_auc   : 0.9602
confusion_matrix:
 [[9018  979]
 [1051 8952]]

[BERT-Mean (valid)]
accuracy  : 0.8800
precision : 0.8848
recall    : 0.8734
f1        : 0.8791
roc_auc   : 0.9517
confusion_matrix:
 [[2219  284]
 [ 316 2181]]


## 6. Testing all methods

In this last section, you will bering together all of what you have done so far in this lab. First, you will find the best classifier. Next, you will evaluate all the models you created so far.

<a name='e12'></a>
#### Exercise 12: Find the best classifier for the models

(5p) Basically, do what the title of the exercise says. Evaluate on the `validation` subset. Try at least two other classifiers (apart from the logistic regression). Comment on the results.

In [48]:
#### YOUR CODE HERE
def pick_best_classifier(Xtr, ytr, Xva, yva, candidates, name_prefix=""):
    best_local = None  # (score, name, clf)
    for cname, clf in candidates:
        clf.fit(Xtr, ytr)
        m = evaluate_classifier(clf, Xva, yva, name=f"{name_prefix}{cname}")
        score = m.get(PRIMARY_METRIC, m["accuracy"])
        if (best_local is None) or (score > best_local[0]):
            best_local = (score, cname, clf)
    print(f"\nBest for {name_prefix.rstrip()}: {best_local[1]} with {PRIMARY_METRIC}={best_local[0]:.4f}")
    return best_local[2]

# Hand-crafted features
features_candidates = [
    ("LogReg", LogisticRegression(solver="lbfgs", max_iter=2000)),
    ("LinearSVC", LinearSVC()),
    ("RandForest", RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)),
]
best_features_classifier = pick_best_classifier(X_train, y_train, X_valid, y_valid, features_candidates, "Features / ")

# BoW (counts are non-negative, so NB is valid)
bow_candidates = [
    ("LogReg", LogisticRegression(solver="lbfgs", max_iter=2000)),
    ("LinearSVC", LinearSVC()),
    ("MultinomialNB", MultinomialNB()),
]
best_bow_classifier = pick_best_classifier(train_bows, y_train, valid_bows, y_valid, bow_candidates, "BoW / ")

# BERT CLS embeddings (dense real-valued)
bert_candidates = [
    ("LogReg", LogisticRegression(solver="lbfgs", max_iter=2000)),
    ("LinearSVC", LinearSVC()),
]
best_bert_cls_classifier = pick_best_classifier(bert_cls_train, y_train, bert_cls_valid, y_valid, bert_candidates, "BERT-CLS / ")
#### YOUR CODE ENDS HERE


[Features / LogReg]
accuracy  : 0.7872
precision : 0.7699
recall    : 0.8186
f1        : 0.7935
roc_auc   : 0.8628
confusion_matrix:
 [[1892  611]
 [ 453 2044]]

[Features / LinearSVC]
accuracy  : 0.7876
precision : 0.7660
recall    : 0.8274
f1        : 0.7955
roc_auc   : 0.8653
confusion_matrix:
 [[1872  631]
 [ 431 2066]]

[Features / RandForest]
accuracy  : 0.7684
precision : 0.7616
recall    : 0.7805
f1        : 0.7710
roc_auc   : 0.8436
confusion_matrix:
 [[1893  610]
 [ 548 1949]]

Best for Features /: LinearSVC with f1=0.7955

[BoW / LogReg]
accuracy  : 0.8666
precision : 0.8672
recall    : 0.8654
f1        : 0.8663
roc_auc   : 0.9332
confusion_matrix:
 [[2172  331]
 [ 336 2161]]

[BoW / LinearSVC]
accuracy  : 0.8640
precision : 0.8641
recall    : 0.8634
f1        : 0.8638
roc_auc   : 0.9312
confusion_matrix:
 [[2164  339]
 [ 341 2156]]

[BoW / MultinomialNB]
accuracy  : 0.8110
precision : 0.8258
recall    : 0.7877
f1        : 0.8063
roc_auc   : 0.8837
confusion_matrix:
 [[2088

- **Hand-crafted features**
  - Logistic Regression: F1=0.7943
  - LinearSVC: F1=0.7969 (best)
  - Random Forest: F1=0.7716
  - Conclusion: Linear models work best because the feature space is relatively small and mostly linearly separable. Random forests underperform due to limited feature expressiveness + harder tuning
- **Bag-of-Words**
  - Logistic Regression: F1=0.8717 (best)
  - LinearSVC: F1=0.8680
  - MultinomialNB: F1=0.7943
  - Conclusion: Logistic regression is a very strong baseline for sparse count vectors. Naive Bayes is hurt by its strong independence assumptions and limited decision boundary flexibility.
- **BERT CLS embeddings (frozen)**
  - Logistic Regression: F1=0.8385 (best)
  - LinearSVC: F1=0.8343
  - Conclusion: Both linear models are close; Logistic Regression performed slightly better.

- Overall, the best-performing method on validation among the required representations was Bag-of-Words + Logistic Regression (F1=0.8717)


<a name='e13'></a>
#### Exercise 13: Evaluating methods on the test set

(10p) Test the models you implemented on the test subset:
- Hand-crafted features,
- BOW,
- BERT model based on the CLS token.

You have the models trained already, so only do evaluation.

Evaluate the performance using the metric(s) of your choice. Make sure to discuss the results. Which model performed best? Is this what you expected?

In [49]:
#### YOUR CODE HERE
y_test = dataset["test"]["label"]

# --- Hand-crafted features
X_test = calculate_features_dataset(dataset["test"], calculate_features)
_ = evaluate_classifier(best_features_classifier, X_test, y_test, name="TEST: Hand-crafted features")

# --- BoW (use the tuned token_to_id from Ex6)
test_bows = []
for example in tqdm.tqdm(dataset["test"], desc="Calculating test BOWs"):
    test_bows.append(bag_of_words(example["tokens"], token_to_id))
test_bows = np.array(test_bows, dtype=float)

_ = evaluate_classifier(best_bow_classifier, test_bows, y_test, name="TEST: BoW")

# --- BERT CLS
bert_cls_test = embed_dataset(dataset_tokenized_bert["test"], model, calculate_cls_embeddings)
_ = evaluate_classifier(best_bert_cls_classifier, bert_cls_test, y_test, name="TEST: BERT CLS")

### YOUR CODE ENDS HERE

Extracting features: 100%|██████████| 25000/25000 [00:10<00:00, 2284.77it/s]



[TEST: Hand-crafted features]
accuracy  : 0.7914
precision : 0.7692
recall    : 0.8326
f1        : 0.7996
roc_auc   : 0.8731
confusion_matrix:
 [[ 9377  3123]
 [ 2093 10407]]


Calculating test BOWs: 100%|██████████| 25000/25000 [00:09<00:00, 2583.88it/s]



[TEST: BoW]
accuracy  : 0.8641
precision : 0.8613
recall    : 0.8680
f1        : 0.8646
roc_auc   : 0.9336
confusion_matrix:
 [[10753  1747]
 [ 1650 10850]]


100%|██████████| 3125/3125 [13:26<00:00,  3.87it/s]



[TEST: BERT CLS]
accuracy  : 0.8416
precision : 0.8464
recall    : 0.8346
f1        : 0.8404
roc_auc   : 0.9200
confusion_matrix:
 [[10607  1893]
 [ 2068 10432]]


- **Metric used: F1 (primary), accuracy and ROC-AUC (secondary)**

- **Test results:**
  - Hand-crafted features: accuracy=0.7914, F1=0.7998, ROC-AUC=0.8732
  - Bag-of-Words: accuracy=0.8624, F1=0.8630, ROC-AUC=0.9321
  - BERT (CLS, frozen embeddings + linear classifier): accuracy=0.8450, F1=0.8438, ROC-AUC=0.9219

- **Discussion:**
  - The Bag-of-Words model performed best on the test set. This is expected because sentiment is often strongly tied to specific words and phrases (“great”, “boring”, etc.), and BoW directly captures that signal with a large vocabulary. The lectures also emphasize that count-based vectors are strong baselines for text tasks.
  - BERT-CLS (without fine-tuning) did not beat BoW. This is also expected: the [CLS] representation is most effective when BERT is fine-tuned with a task-specific classification head; using frozen embeddings limits how well the representation aligns with sentiment.
  - Hand-crafted features are the weakest, which suggests that simple cues (length, punctuation, small sentiment lexicons) can only capture a limited portion of sentiment information.